# TP 2a: SPATIALLY EXTENDED WILSON-COWAN MODEL / ACTIVE TRANSIENT
## Contact: emre.baspinar@inria.fr

In [ ]:
#####################################################################
## TP 2a: SPATIALLY EXTENDED WILSON-COWAN MODEL / ACTIVE TRANSIENTS ##
#####################################################################

# Contact: emre.baspinar@inria.fr

#####################################################################
## Initialization #################################################
#####################################################################
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib.animation import FFMpegWriter
from scipy.signal import convolve
#---------------------------------------------------------------------#
#---------------------------------------------------------------------#

## Population transfer functions

In [ ]:
#########################################################################################
## Population transfer functions: sigmoids ##############################################
#########################################################################################

# Define the excitatory population transfer function
def S_e(x, beta_e, xi0_e):
    return 1 / (1 + np.exp(-beta_e * (x - xi0_e)))

# Define the inihbitory population transfer function
def S_i(x, beta_i, xi0_i):
    return 1 / (1 + np.exp(-beta_i * (x - xi0_i)))

## Plot the transfer functions

# Parameters of the population transfer functions
beta_exc = 0.5         # nonlinearity sharpness of excitatory population transfer function
beta_inh = 0.3         # nonlinearity sharpness of inhibitory population transfer function
xi0_exc = 9.0          # activity threshold of excitatory population transfer function
xi0_inh = 17.0         # activity threshold of inhibitory population transfer function

xVector = np.linspace(-5, 40, 500)  # x-vector for plotting

# Plot the excitatory population transfer function S_e
plt.figure(figsize=(8, 4))
plt.plot(xVector, S_e(xVector, beta_e=beta_exc, xi0_e=xi0_exc))
plt.title("Connectivity Kernel")
plt.xlabel("x")
plt.ylabel("Kernel value")
plt.grid(False)
plt.tight_layout()
plt.show()

# Plot the inhibitory population transfer function S_i
plt.figure(figsize=(8, 4))
plt.plot(xVector, S_i(xVector, beta_i=beta_inh, xi0_i=xi0_inh))
plt.title("Connectivity Kernel")
plt.xlabel("x")
plt.ylabel("Kernel value")
plt.grid(False)
plt.tight_layout()
plt.show()

## Connectivity kernel
## Step 1
What does the connectivity kernel represent in the spatialy-extended Wilson-Cowan system? Why do we choose it as a decaying function?

Change the "sigma" parameter in the connectivity kernel and comment on the results. What does the sigma parameter in the connectivity kernel represent? How can you interpret it in a biological context (long-range interactions, short-range interactions)?

In [ ]:
# Define generic connectivity kernel: Gaussian
def connectivity_kernel_gaussian(x, amplitude, sigma):
    return amplitude * np.exp(-np.abs(x)**2 / sigma)

# Define generic connectivity kernel: exponential
def connectivity_kernel_exponential(x, amplitude, sigma):
    return amplitude * np.exp(-np.abs(x) / sigma)

# Parameters for the Gaussian type
amplitude = 1.0      # amplitude
sigma = 0.2     # scale
xVector = np.linspace(-10, 10, 500)  # x-vector for plotting

# Plot the Gaussian type
plt.figure(figsize=(8, 4))
plt.plot(xVector, connectivity_kernel_gaussian(xVector, amplitude, sigma))
plt.title("Connectivity Kernel")
plt.xlabel("x")
plt.ylabel("Kernel value")
plt.grid(False)
plt.tight_layout()
plt.show()

# Plot the exponential type
plt.figure(figsize=(8, 4))
plt.plot(xVector, connectivity_kernel_exponential(xVector, amplitude, sigma))
plt.title("Connectivity Kernel")
plt.xlabel("x")
plt.ylabel("Kernel value")
plt.grid(False)
plt.tight_layout()
plt.show()

## Step 2
Change the amplitude parameter in the connectivity kernels and comment on the results. What do you observe? What these changes represent as we change the amplitude?

In [ ]:
# Parameters for the Gaussian type
amplitude = 1.0      # amplitude
sigma = 1     # scale
xVector = np.linspace(-10, 10, 500)  # x-vector for plotting

# Plot the Gaussian type
plt.figure(figsize=(8, 4))
plt.plot(xVector, connectivity_kernel_gaussian(xVector, amplitude, sigma))
plt.title("Connectivity Kernel")
plt.xlabel("x")
plt.ylabel("Kernel value")
plt.grid(False)
plt.tight_layout()
plt.show()

# Plot the exponential type
plt.figure(figsize=(8, 4))
plt.plot(xVector, connectivity_kernel_exponential(xVector, amplitude, sigma))
plt.title("Connectivity Kernel")
plt.xlabel("x")
plt.ylabel("Kernel value")
plt.grid(False)
plt.tight_layout()
plt.show()

## External stimulus
## Step 3
What are the parameters "P_amplitude", "Q_amplitude", "x0", "stimWidth" and stim_dur represent? Change these parameters one by one and comment on the results.

In [ ]:
# External stimuli
P_amplitude = 4.7     # amplitude of external stimulus P
Q_amplitude = 0.0     # amplitude of external stimulus Q
x0 = 0.0              # spatial center of input
stimWidth = 80        # spatial width of input 
t1 = 0.0              # stimulus start time (ms)
stim_dur = 5          # stimulus duration (ms)
# t2 = t1 + stim_dur  # stimulus end time (ms)
t2 =  t1+stim_dur     # stimulus end time (ms)

N = 501
L = 1000
xVector = np.linspace(-L/2, L/2, N)  # x-vector for plotting

# Compute the spatial profile of P
P_spatial = np.where(np.abs(xVector - x0) <= stimWidth/2, P_amplitude, 0)
P_stimulus = P_spatial

# Q is zero!
Q_stimulus = Q_amplitude * np.ones(xVector.shape[0])

# Plot the external stimulus P
plt.figure(figsize=(8, 4))
plt.plot(xVector, P_spatial)
plt.title("External stimulus P")
plt.xlabel("x")
plt.ylabel("Stimulus value")
plt.grid(False)
plt.tight_layout()
plt.show()

# Plot the external stimulus Q
plt.figure(figsize=(8, 4))
plt.plot(xVector, Q_stimulus)
plt.title("External stimulus Q")
plt.xlabel("x")
plt.ylabel("Stimulus value")
plt.grid(False)
plt.tight_layout()
plt.show()


## Simulations
## Step 4
a) We define below the simulation parameters regarding the spatial discretization and integration in time. What do we mean by "spatial grid"? What are the parameters "L", "N", "dx" and "x" for the spatial grid represent? Why do we need a spatial grid for the simulations?

b) There are two different categories of parameters which we did not have in the localized Wilson-Cowan model. They are the parameters regarding the connectivity kernels. What do they determine in the connectivity kernels?

c) Perform the simulation and comment on the result. What is the particularity of the activity pattern (active transient) which we observe? 

d) Decrease the number of spatial samples (N) and comment on the results.

e) Fix the number of spatial samples N to 501. Change the connectivity kernel from the exponential to the Gaussian type. What do you observe? Comment on the result.

In [ ]:

########################################################################################
## Simulation parameters ###############################################################
########################################################################################

# Spatial grid
L = 1000                        # domain size (=1000 \mu m)
N = 501                         # number of grid points (odd number for perfect spatial symmetry)
dx = L / N                  # spatial step size
x = np.linspace(-L/2, L/2, N)   # discretized spatial domain

# Time integration
dt = 0.025                      # time step size
T = 50                          # final time
steps = int(T / dt)             # number of steps in time

#---------------------------------------------------------------------------------------#
#---------------------------------------------------------------------------------------#


#########################################################################################
##  Model parameters: active transient  #################################################
#########################################################################################

# Wilson-Cowan parameters
mu = 10              # time constant for E and I
re = 1               # refractory period for excitatory neurons
ri = 1               # refractory period for inhibitory neurons

# Maximum amplitudes of the connectivity kernels
amplitudeSet = {
    'ee': 1.5,
    'ie': 1.35,
    'ei': 1.35,
    'ii': 1.8
}

# Scale values of the connectivity kernels
sigmaSet = {
    'ee': 40,
    'ie': 60,
    'ei': 60,
    'ii': 30
}

## Assign the parameters to the connectivity kernels for each population pair. BE CAREFUL! THESE ARE NOT COEFFICIENTS BUT KERNELS. IT IS DIFFERENT FROM THE LOCALIZED WILSON-COWAN CONNECTIVITY WEIGHTS!!!
## To use the Gaussian type kernel, uncomment below the lines with "connectivity_kernel_gaussian"
wee = connectivity_kernel_exponential(x, amplitudeSet['ee'], sigmaSet['ee']) # excitatory to excitatory
wei = connectivity_kernel_exponential(x, amplitudeSet['ei'], sigmaSet['ei']) # inhibitory to excitatory
wie = connectivity_kernel_exponential(x, amplitudeSet['ie'], sigmaSet['ie']) # excitatory to inhibitory
wii = connectivity_kernel_exponential(x, amplitudeSet['ii'], sigmaSet['ii']) # inhibitory to inhbitory
# wee = connectivity_kernel_gaussian(x, amplitudeSet['ee'], sigmaSet['ee']) # excitatory to excitatory
# wei = connectivity_kernel_gaussian(x, amplitudeSet['ei'], sigmaSet['ei']) # inhibitory to excitatory
# wie = connectivity_kernel_gaussian(x, amplitudeSet['ie'], sigmaSet['ie']) # excitatory to inhibitory
# wii = connectivity_kernel_gaussian(x, amplitudeSet['ii'], sigmaSet['ii']) # inhibitory to inhbitory

#---------------------------------------------------------------------------------------#
#---------------------------------------------------------------------------------------#

# Set the initial conditions for the populations
E = np.zeros(N)
I = np.zeros(N)


# External stimuli
P_spatial = np.where(np.abs(x - x0) <= stimWidth/2, P_amplitude, 0)
P_stimulus = P_spatial

# Q is zero!
Q_stimulus = Q_amplitude * np.ones(x.shape[0])

#########################################################################################
## Simulation of the model ##############################################################
#########################################################################################

# Choose the discretization node index for time plots
x_target = 0                                   # central node
idx_target = np.argmin(np.abs(x - x_target))

# Initialize a vector for the time series of the
# excitatory population
E_time_series = []

# Initialize a vector for the samples of the time series
# of the excitatory population for animated spatial plots
E_record = []
recordTimeStep = 0.1
record_interval = int(recordTimeStep / dt)
recordTimeVector = []

# Initialize a vector for the samples of the time series
# of the excitatory external stimulus P
P_record = []

## Integrate the model in time
for tIndex in range(steps):
    currentTime = tIndex * dt

    # Define time-varying excitatory external stimulus P(x, t)
    if t1 <= currentTime <= t2:
        P_stimulus = P_spatial
    else:
        P_stimulus = np.zeros_like(x)

    # Synaptic interactions
    conv_ee = convolve(E, wee, mode='same') * dx
    conv_ei = convolve(I, wei, mode='same') * dx
    conv_ie = convolve(E, wie, mode='same') * dx
    conv_ii = convolve(I, wii, mode='same') * dx

    # Output of the transfer functions
    Fe = S_e(conv_ee - conv_ei + P_stimulus, beta_e=beta_exc, xi0_e=xi0_exc)
    Fi = S_i(conv_ie - conv_ii + Q_stimulus, beta_i=beta_inh, xi0_i=xi0_inh)

    # Compute the increments of E and I in time  
    dE = (-E + (1 - re * E) * Fe) / mu
    dI = (-I + (1 - ri * I) * Fi) / mu

    # Compute E(x,t) and I(x,t) for all x in the domain
    E += dt * dE
    I += dt * dI

    # Save E(x,t)
    E_time_series.append(E[idx_target])

    # Save E(x,t) and P(x,t) sampled with sample step size = record_interval
    if tIndex % record_interval == 0:
        E_record.append(E.copy())
        P_record.append(P_stimulus.copy())
        recordTimeVector.append(currentTime)

#---------------------------------------------------------------------------------------#
#---------------------------------------------------------------------------------------#


#########################################################################################
## Plot the simulation results ##########################################################
#########################################################################################

## Plot the central discretization node in time
# Time vector
time = np.arange(steps) * dt

# Plotting E(x=x_target, t) (x_target: central node index)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time, E_time_series, label='E(t)')

# Vertical dashed line
ax.axvline(x=t2, color='k', linestyle='--', linewidth=1)

# Highlight the time on the x-axis
ax.text(t2, -0.05, f'{t2}',
        ha='center', va='top',
        transform=ax.get_xaxis_transform())

plt.title('Time evolution of E at x = %i' %x_target+" (amplitude of P = %1.1f)" %P_amplitude)
plt.xlabel('Time [ms]')
plt.ylabel('E(x=%i, t)' %x_target)
plt.xlim(0, T)
plt.grid(False)
plt.tight_layout()
plt.show()

## Plot and save the time evolution of the spatially extended excitatory population
# Create the figure and axis
fig, ax = plt.subplots()
line_E, = ax.plot(x, E_record[0], lw=2, label="E_record", color="blue")
line_P, = ax.plot(x, P_record[0], lw=2, label="P_record", color="red")
# ax.set_ylim(np.min(E_record), np.max(E_record))
if P_amplitude > 1:
    ax.set_ylim(0, P_amplitude+0.1)
else:
    ax.set_ylim(0, 1)

ax.set_xlim(x[0], x[-1])
ax.set_xlabel("Space (x)")
ax.set_ylabel("Excitatory activity (E) and external stimulus (P)")
title = ax.set_title("Time: 0.00")
ax.legend()

# Animation update function
def update(frame):
    line_E.set_ydata(E_record[frame])
    line_P.set_ydata(P_record[frame])
    title.set_text(f"Time: {recordTimeVector[frame]:.2f}")
    return line_E, line_P, title

## Create and show the animation
anim = FuncAnimation(fig=fig, func=update, frames=len(recordTimeVector), interval=50, blit=False)
plt.close(fig)

# Display in Jupyter or VS Code notebook -- WARNING: IF IT DOES NOT WORK, SAVE THE ANIMATION BY UNCOMMENTING BELOW!
HTML(anim.to_html5_video())

# ## Create and save the animation
# anim = FuncAnimation(fig=fig, func=update, frames=len(recordTimeVector), interval=500, blit=False)
# plt.close(fig)

# # Save the animation -- WARNING: USE IT IF THE HTML CODE ABOVE DOES NOT WORK.
# # You will find the animation mp4 file in the same folder where the Juyter notebook is located.
# writer = FFMpegWriter(fps=30)
# anim.save("animationActiveTransient.mp4", writer=writer)


#---------------------------------------------------------------------------------------#
#---------------------------------------------------------------------------------------#

#########################################################################################
######################################## THE END ########################################
#########################################################################################